# Day 03: Scopes, Error Handling, Modules, and Packages

This notebook practices name lookup (scope), exceptions, modules, and packages. Predict each result before running the examples, then explain the behavior in the reflection prompts.

**Package terminology:** Python “package” can mean a folder organizing importable modules, or installable software such as pytest. This notebook distinguishes the two.

Choose the project `.venv` kernel in VS Code.

## 1. Scope and namespace

A **namespace** maps names to objects. A **scope** is where a name can be used directly. The normal name lookup order in a function is **LEGB**: Local, Enclosing function scopes, Global module scope, then Built-ins. Assignment inside a function normally binds a local name.

In [ ]:
message = "module/global"

def show_message():
    message = "local"
    print(message)

show_message()
print(message)

**Trace both lookups:** Why do the two prints produce different values even though they use the same spelling?

> Write here.

---

## 2. Enclosing scope and `nonlocal`

A nested function can read names from an enclosing function. To rebind an enclosing function's name, declare it `nonlocal`. It targets the nearest enclosing function binding, not the module global.

In [ ]:
def make_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter = make_counter()
print(counter())
print(counter())

**Explain:** Where is `count` first bound? Which function changes that binding? Why does it still exist after `make_counter()` returns?

> Write here.

### `global`

`global name` tells a function that assignments to that name should rebind the module-level name. This differs from mutating a global list or dictionary: mutation changes the object, not the name's binding.

In [ ]:
status = "new"

def mark_done():
    global status
    status = "done"

mark_done()
print(status)

**In your words:** How does `global` differ from `nonlocal`? When is passing a value into a function clearer than changing a global?

> Write here.

---

## 3. Local assignment, shadowing, and `UnboundLocalError`

Python determines a function's local names from assignments in that function. If a name is assigned anywhere in a function, using it earlier in that function can produce `UnboundLocalError`, even when a global with that name exists. Uncomment the commented print to observe it.

In [ ]:
value = 10

def read_then_assign():
    # Uncomment the next line to trigger UnboundLocalError.
    # print(value)
    value = 20
    return value

print(read_then_assign())

**Why would uncommenting `print(value)` make Python treat `value` as local throughout the function?**

> Write here.

---

## 4. Errors and exceptions

A **syntax error** means Python cannot parse the source code. An **exception** occurs while syntactically valid code runs. An unhandled exception displays a traceback. Start by reading the exception type and the last relevant lines.

In [ ]:
# Uncomment one example at a time and observe its exception type.
# int("twelve")       # ValueError
# 10 / 0              # ZeroDivisionError
# missing_name        # NameError
# [1, 2][5]           # IndexError
print("The error examples are commented out.")

### Handle expected failures with `try` / `except`

Keep the `try` block focused on the operation that can fail. Catch the specific exception you expect and take a useful action. Avoid bare `except:` or catching every exception without a recovery plan.

In [ ]:
raw_age = "unknown"
try:
    age = int(raw_age)
except ValueError as error:
    print(f"Please provide a whole number: {error}")

**Practice:** Change `raw_age` to `"42"`. Add an `else` block that prints the parsed age only if conversion succeeds.

> Write or sketch your change here.

### `else`, `finally`, and `raise`

`else` runs only if the `try` block completed without an exception. `finally` runs whether the operation succeeded or failed, often for cleanup. `raise` signals an exception when a function cannot fulfill its contract.

In [ ]:
def positive_amount(text):
    try:
        amount = float(text)
    except ValueError:
        raise ValueError("amount must be numeric")
    else:
        if amount < 0:
            raise ValueError("amount must not be negative")
        return amount
    finally:
        print("Finished amount check")

print(positive_amount("12.5"))

**Predict first:** What does the `finally` block print for valid input? What if the call uses `"abc"`?

> Write here.

---

## 5. Modules and imports

A **module** is a Python file, typically ending in `.py`, that contains reusable code. Importing a module makes its namespace available. Prefer explicit imports such as `import math` or `from pathlib import Path`; avoid `from module import *` because it hides where names came from.

In [ ]:
import math
from pathlib import Path
import json as json_module

print(math.sqrt(81))
print(Path("notes").name)
print(json_module.dumps({"ok": True}))

### Make and import a small module

The next cell writes `greetings.py` into a dedicated practice folder, adds that folder to this notebook's import search path for the session, and imports it.

In [ ]:
from pathlib import Path
import sys

module_dir = Path("day3_import_practice")
module_dir.mkdir(exist_ok=True)
(module_dir / "greetings.py").write_text(
    "def greet(name):\n    return f'Hello, {name}!'\n",
    encoding="utf-8",
)
sys.path.insert(0, str(module_dir.resolve()))

import greetings
print(greetings.greet("Learner"))

**What is the module name in the import? What is the file name? Why is `.py` omitted?

> Write here.

### `__name__` and script entry points

A module's `__name__` is `"__main__"` when that file is run as the program entry point. When imported, it gets its module name. The common guard keeps script-only behavior out of imports.

In [ ]:
# Typical pattern inside a reusable .py file:
def main():
    print("Run the program")

if __name__ == "__main__":
    main()

**Why is this guard useful when a module can be both run and imported?**

> Write here.

---

## 6. Packages: folders of modules

A package organizes related modules in a directory. A regular beginner package commonly has an `__init__.py` file:

```text
shop/
    __init__.py
    prices.py
    orders.py
```

`shop/prices.py` is the `shop.prices` module. Use `import shop.prices` or `from shop.prices import ...`. Inside a package, relative imports such as `from . import prices` refer to sibling modules.

### Package sketch exercise

Imagine a package named `orders_app` with modules `validation.py` and `convert.py`.

1. Draw the folder structure, including `__init__.py`.
2. Write an import in `convert.py` to use `validate` from `validation.py`.
3. Write an import that a separate `run.py` could use.

> Write your structure and imports here.

Modern Python also supports namespace packages without `__init__.py`; use the regular package layout for this beginner exercise.

### Packages as installed distributions

An installable **distribution package** is software installed into a Python environment, commonly from PyPI. `pip` installs a distribution; Python code imports a module or package name. Install names and import names may differ. This project environment already has pytest and DuckDB installed.

In [ ]:
# This prints the Python interpreter/environment selected for this notebook.
import sys
print(sys.executable)

# In a terminal, an install commonly looks like:
# python -m pip install distribution-name
# In Python code, you then import a module, for example:
import json

**Explain the distinction:** What does `pip install` do? What does `import` do?

> Write here.

---

## 7. Combine functions, scope, and exceptions

This example validates one CSV-like row. It combines function-local names, type conversion, exception handling, and a return value. It does not read or write files.

In [ ]:
def validate_amount(row):
    try:
        amount = float(row["amount"])
    except (KeyError, ValueError):
        return None, "amount is missing or not numeric"

    if amount < 0:
        return None, "amount must be non-negative"

    clean_row = {**row, "amount": amount}
    return clean_row, None

print(validate_amount({"order_id": "7", "amount": "4.50"}))
print(validate_amount({"order_id": "8", "amount": "bad"}))

**Explain:** Which names are local to `validate_amount`? Which exceptions are handled? What does each returned tuple mean?

> Write here.

---

## 8. Review checkpoint

- [ ] I can explain LEGB and distinguish local, enclosing, and global names.
- [ ] I know when `global` and `nonlocal` affect rebinding.
- [ ] I can distinguish a syntax error from a runtime exception.
- [ ] I can choose a specific exception to catch and use `else` / `finally` appropriately.
- [ ] I can explain what a module is and how imports work.
- [ ] I recognize the `if __name__ == "__main__"` pattern.
- [ ] I can sketch a package and import between its modules.
- [ ] I can distinguish installing a distribution with `pip` from importing code.

**What needs another example before the P1 converter?**

> Write here.

**Confidence after Day 3 study (1–5):** ___ / 5